# Example 11.4.2 / 3: Tighten Relaxations of Nonlinear Terms

Companion notebook for **11.4.2 Mixed-Integer Nonlinear Models** in the AMPL lab Documentation, section "Tighten Relaxations of Nonlinear Terms". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Tighten Relaxations of Nonlinear Terms

Global solvers bound a nonconvex term by replacing it with a convex relaxation over the variables'
bounds, so the strength of the bound depends on those bounds. Three common devices, each defined in
one line:

- **McCormick inequalities**: four linear inequalities that bound a product $w=xy$ from below and
  above using the bounds of $x$ and $y$. Over a rectangle they form the **convex hull** of the
  product, the smallest convex set containing all its points, but they remain a relaxation: they
  admit values of $w$ other than $xy$.
- **Perspective formulation**: for a convex cost $f(x)$ that applies only when a binary $z$ is 1,
  using $z\,f(x/z)$ instead of $f(x)$ gives a tighter relaxation; at $z=0$ it is defined by its
  limit, which forces $x=0$.
- **Nonlinear large-M implication**: $g(x)\le b+M(1-y)$ switches a constraint off when $y=0$, valid
  only when $M$ bounds $g(x)-b$ over all feasible $x$.

Worked instances:

| Technique | Instance | Result |
|---|---|---|
| McCormick envelope | $x\in[0,2]$, $y\in[0,3]$, point $(1,1.5)$ where $xy=1.5$ | $w\ge0$, $w\ge3x+2y-6=0$, $w\le3x=3$, $w\le2y=3$: the relaxation allows $w\in[0,3]$ |
| tighter bounds | $x\in[0.5,1.5]$, $y\in[1,2]$, same point | $w\ge1.25$ from both lower inequalities, $w\le1.75$ from both upper ones: $[1.25,1.75]$ |
| perspective | on/off cost $x^2$ with $x\le z$, relaxation point $x=1$, $z=0.5$ | plain term gives $1$; perspective $x^2/z$ gives $2$, a tighter value; at $z=0$ the closed extension forces $x=0$ |
| nonlinear large-M | $x^2\le4+M(1-y)$ with $x\in[0,5]$ | valid only for $M\ge25-4=21$; $M=10$ wrongly forbids $x=5$ when $y=0$, since $25>14$ |

The two McCormick intervals from an LP solve, minimizing and maximizing `W` at the fixed point:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param xL; param xU; param yL; param yU;  # bounds of x, then of y
param x0; param y0;                      # point where the product x0 * y0 is relaxed

var W;  # stands for the product x * y

subject to Lower1: W >= xL * y0 + x0 * yL - xL * yL;  # McCormick under-estimator at (xL, yL)
subject to Lower2: W >= xU * y0 + x0 * yU - xU * yU;  # McCormick under-estimator at (xU, yU)
subject to Upper1: W <= xU * y0 + x0 * yL - xU * yL;  # McCormick over-estimator at (xU, yL)
subject to Upper2: W <= xL * y0 + x0 * yU - xL * yU;  # McCormick over-estimator at (xL, yU)

minimize Low: W;   # lower end of the interval for W
maximize High: W;  # upper end of the interval for W

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["x0"] = 1
ampl.param["y0"] = 1.5

In [5]:
# run.py
for box in ((0, 2, 0, 3), (0.5, 1.5, 1, 2)):
    for name, value in zip(("xL", "xU", "yL", "yU"), box):
        ampl.param[name] = value
    ends = []
    for objective in ("Low", "High"):
        ampl.eval(f"objective {objective};")
        ampl.solve(solver="highs", verbose=False)
        ends.append(ampl.var["W"].value())
    print(box, ends)

(0, 2, 0, 3) [0.0, 3.0]
(0.5, 1.5, 1, 2) [1.25, 1.75]
